# FlyRank Search Intelligence Capstone

## Which search-performance signals are associated with content decline, and can they support a transparent review queue?

This capstone studies a practical Search Intelligence question: **which observable signals are associated with a future decline in search visibility, and can those signals be converted into a useful, explainable ranking system?**

The analysis compares a transparent hand-written baseline with a machine-learning model on the **same client-grouped holdout**. The score is used for decision support, not as proof of Google's ranking algorithm or proof that a refresh will cause a causal improvement.

## 1. Data contract and leakage boundary

For the full warehouse, the preferred setup uses DuckDB to read the gated `FlyRank/internship-warehouse` Parquet release without downloading the whole warehouse. A feature window is built from the 30 days immediately before the decision boundary, and the following 30 days are used only to define the outcome. Query-level signals are aggregated from the 90-day query table.

For a reproducible public run when the gated Hugging Face token is not available, the notebook falls back to the public starter CSV. In that mode, the existing `trend_direction` field is used only as the observed label and never as a model feature. The output explicitly records which data mode was used.

In [ ]:
from pathlib import Path
import os
import json
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

RANDOM_STATE = 42
ROOT = Path.cwd()
if not (ROOT / 'work').exists():
    ROOT = Path('..').resolve()
(ROOT / 'paper').mkdir(parents=True, exist_ok=True)
(ROOT / 'work' / 'outputs').mkdir(parents=True, exist_ok=True)

HF_TOKEN = os.environ.get('HF_TOKEN')
FORCE_STARTER = os.environ.get('CAPSTONE_USE_STARTER', '').lower() == '1'
DATA_PATH = os.environ.get('FLYRANK_DATA_PATH')

print('Repository root:', ROOT)
print('Full-warehouse token available:', bool(HF_TOKEN))
print('Starter override:', FORCE_STARTER)

## 2. Build the analysis table

The preferred full-warehouse path uses February/March-style historical windows around the latest available date: prior 30 days for features and the next 30 days for the decline label. This keeps the target in the future relative to the features.

The public starter path uses the same spirit of the task with the already prepared 30,000-row content-refresh slice.

In [ ]:
mode = 'starter'
data = None

if HF_TOKEN and not FORCE_STARTER:
    try:
        import duckdb
        con = duckdb.connect()
        # Keep the token out of SQL text by passing it through a DuckDB session variable.
        con.execute('SET VARIABLE hf_token = ?', [HF_TOKEN])
        con.execute("CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN getvariable('hf_token'))")
        REL = 'hf://datasets/FlyRank/internship-warehouse'
        FACT = f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"
        QUERY90 = f"read_parquet('{REL}/fact_content_query_90d.parquet')"

        bounds = con.sql(f"SELECT MAX(report_date) AS max_date FROM {FACT}").df()
        end_d = pd.to_datetime(bounds.loc[0, 'max_date'])
        start_d = end_d - pd.Timedelta(days=60)
        feature_cut = end_d - pd.Timedelta(days=30)

        daily = con.sql(f"""
            SELECT client_hash_id, content_hash_id,
                   SUM(CASE WHEN report_date > DATE '{start_d.date()}' AND report_date <= DATE '{feature_cut.date()}' THEN gsc_impressions ELSE 0 END) AS imp_prev30,
                   SUM(CASE WHEN report_date > DATE '{start_d.date()}' AND report_date <= DATE '{feature_cut.date()}' THEN gsc_clicks ELSE 0 END) AS clk_prev30,
                   AVG(CASE WHEN report_date > DATE '{start_d.date()}' AND report_date <= DATE '{feature_cut.date()}' THEN gsc_avg_position END) AS pos_prev30,
                   SUM(CASE WHEN report_date > DATE '{feature_cut.date()}' AND report_date <= DATE '{end_d.date()}' THEN gsc_impressions ELSE 0 END) AS imp_next30
            FROM {FACT}
            WHERE report_date > DATE '{start_d.date()}' AND report_date <= DATE '{end_d.date()}'
            GROUP BY 1,2
            HAVING imp_prev30 >= 100
        """).df()

        qsignals = con.sql(f"""
            SELECT content_hash_id,
                   ANY_VALUE(content_visible_query_count) AS visible_queries,
                   ANY_VALUE(rare_impressions_share) AS rare_share,
                   ANY_VALUE(anonymized_impressions_share) AS anon_share,
                   MAX(impressions_90d) AS top_query_impressions,
                   SUM(impressions_90d) AS kept_impressions
            FROM {QUERY90}
            GROUP BY content_hash_id
        """).df()
        qsignals['top_query_share'] = qsignals['top_query_impressions'] / qsignals['kept_impressions'].replace(0, np.nan)
        data = daily.merge(qsignals, on='content_hash_id', how='left')
        data['target'] = (data['imp_next30'] < 0.8 * data['imp_prev30']).astype(int)
        mode = 'full_warehouse'
        decision_date = str(end_d.date())
        print(f'Full warehouse mode: {len(data):,} content items; decision boundary: {decision_date}')
    except Exception as exc:
        print('Full-warehouse path was not available; using the public starter slice.')
        print('Reason:', type(exc).__name__, str(exc)[:240])
        data = None

if data is None:
    if DATA_PATH:
        starter_path = Path(DATA_PATH)
    else:
        candidates = [
            ROOT / 'data' / 'raw' / 'content_refresh_anonymized.csv',
            Path('data/raw/content_refresh_anonymized.csv')
        ]
        starter_path = next((p for p in candidates if p.exists()), None)
    if starter_path is None:
        raise FileNotFoundError('Starter CSV not found. Set FLYRANK_DATA_PATH or provide data/raw/content_refresh_anonymized.csv.')
    raw = pd.read_csv(starter_path)
    raw = raw[(raw['impressions_90d'] > 0) & (raw['content_age_days'] >= 90)].drop_duplicates('content_id').copy()
    raw['target'] = raw['trend_direction'].astype(str).str.lower().eq('down').astype(int)
    raw['visible_queries'] = np.nan
    raw['rare_share'] = np.nan
    raw['anon_share'] = np.nan
    raw['top_query_share'] = np.nan
    raw['imp_prev30'] = raw['impressions_90d']
    raw['clk_prev30'] = raw['clicks_90d']
    raw['pos_prev30'] = raw['avg_position']
    data = raw
    mode = 'starter'
    decision_date = 'starter-slice-observation'
    print(f'Starter mode: {len(data):,} rows')

print('Positive/decline rate:', f"{data['target'].mean():.2%}")
print('Columns available:', len(data.columns))

## 3. Signal analysis

Two signals are checked before modeling. First, freshness/staleness is compared with the observed decline rate. Second, low CTR among pages that already have search visibility is compared with higher CTR. These checks are descriptive associations, not causal claims.

In [ ]:
if mode == 'starter':
    fresh = pd.cut(data['content_age_days'], bins=[89,180,365,10_000], labels=['90–180 days','181–365 days','366+ days'])
    freshness_table = data.assign(bucket=fresh).groupby('bucket', observed=False).agg(n=('target','size'), decline_rate=('target','mean')).reset_index()
    data['ctr_calc'] = data['clicks_90d'] / data['impressions_90d'].replace(0, np.nan)
    pos_band = pd.cut(data['avg_position'], bins=[0,10,20,50,10_000], labels=['1–10','11–20','21–50','50+'])
    ctr_bucket = np.where(data['ctr_calc'] < 0.005, 'CTR <0.5%', 'CTR ≥0.5%')
    ctr_table = data.assign(position_band=pos_band, ctr_bucket=ctr_bucket).query("position_band in ['1–10','11–20']").groupby(['position_band','ctr_bucket'], observed=False).agg(n=('target','size'), decline_rate=('target','mean')).reset_index()
else:
    data['freshness_days'] = data.get('days_since_last_update', np.nan)
    if data['freshness_days'].notna().any():
        fresh = pd.cut(data['freshness_days'], bins=[-1,90,180,10_000], labels=['0–90 days','91–180 days','181+ days'])
        freshness_table = data.assign(bucket=fresh).groupby('bucket', observed=False).agg(n=('target','size'), decline_rate=('target','mean')).reset_index()
    else:
        freshness_table = pd.DataFrame(columns=['bucket','n','decline_rate'])
    data['ctr_calc'] = data['clk_prev30'] / data['imp_prev30'].replace(0, np.nan)
    pos_band = pd.cut(data['pos_prev30'], bins=[0,10,20,50,10_000], labels=['1–10','11–20','21–50','50+'])
    ctr_bucket = np.where(data['ctr_calc'] < 0.005, 'CTR <0.5%', 'CTR ≥0.5%')
    ctr_table = data.assign(position_band=pos_band, ctr_bucket=ctr_bucket).query("position_band in ['1–10','11–20']").groupby(['position_band','ctr_bucket'], observed=False).agg(n=('target','size'), decline_rate=('target','mean')).reset_index()

print('\nFreshness signal table')
display(freshness_table)
print('\nCTR versus position signal table')
display(ctr_table)

signal_summary = {
    'freshness': 'Directional association observed' if len(freshness_table) else 'Not available',
    'ctr_position': 'Lower CTR has higher observed decline in visible bands' if len(ctr_table) else 'Not available'
}
print(signal_summary)

## 4. Baseline and model

The baseline is deliberately simple. It scores pages using prior visibility, query breadth, and query concentration. The machine-learning model is a Random Forest using the same feature family. Both rankings are evaluated on the same client-grouped holdout so the comparison is fairer than reporting a model on one split and a baseline on another.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

if mode == 'full_warehouse':
    feature_cols = ['imp_prev30','clk_prev30','pos_prev30','visible_queries','rare_share','anon_share','top_query_share']
    group_col = 'client_hash_id'
    id_col = 'content_hash_id'
else:
    feature_cols = ['impressions_90d','clicks_90d','avg_position','ctr_calc','content_age_days','days_since_last_update','word_count']
    group_col = 'client_id'
    id_col = 'content_id'

feature_cols = [c for c in feature_cols if c in data.columns]
model_df = data.dropna(subset=feature_cols + ['target', group_col]).copy()
model_df = model_df.drop_duplicates(id_col)

splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_STATE)
train_idx, test_idx = next(splitter.split(model_df, model_df['target'], groups=model_df[group_col]))
train = model_df.iloc[train_idx].copy()
test = model_df.iloc[test_idx].copy()

X_train, y_train = train[feature_cols], train['target']
X_test, y_test = test[feature_cols], test['target']

model = RandomForestClassifier(
    n_estimators=300,
    min_samples_leaf=8,
    class_weight='balanced_subsample',
    random_state=RANDOM_STATE,
    n_jobs=-1
).fit(X_train, y_train)
model_prob = model.predict_proba(X_test)[:,1]

def pct_rank(s):
    return s.rank(method='average', pct=True).fillna(0)

if mode == 'full_warehouse':
    baseline = 0.50 * (1 - pct_rank(test['visible_queries'])) + 0.30 * pct_rank(test['top_query_share']) + 0.20 * (1 - pct_rank(test['imp_prev30']))
else:
    baseline = (0.40 * pct_rank(test['impressions_90d']) +
                0.30 * pct_rank(test['days_since_last_update']) +
                0.20 * (1 - pct_rank(test['avg_position'].clip(lower=1, upper=50))) +
                0.10 * (1 - pct_rank(test['word_count'])))

def precision_at(y, score, k):
    k = min(k, len(y))
    order = np.argsort(-np.asarray(score))[:k]
    return float(np.asarray(y)[order].mean()) if k else np.nan

metrics = {
    'test_rows': int(len(test)),
    'test_base_rate': float(y_test.mean()),
    'baseline_precision_at_50': precision_at(y_test, baseline, 50),
    'model_precision_at_50': precision_at(y_test, model_prob, 50),
    'baseline_precision_at_100': precision_at(y_test, baseline, 100),
    'model_precision_at_100': precision_at(y_test, model_prob, 100),
    'baseline_average_precision': float(average_precision_score(y_test, baseline)),
    'model_average_precision': float(average_precision_score(y_test, model_prob)),
    'model_roc_auc': float(roc_auc_score(y_test, model_prob)),
}
print(json.dumps(metrics, indent=2))

## 5. Explainability and ranked recommendations

The model ranking is paired with human-readable reasons. The purpose is to create a review queue, not to automate a content change without inspection.

In [ ]:
scored = test.copy()
scored['model_score'] = model_prob
scored['baseline_score'] = np.asarray(baseline)

def make_reason(row):
    reasons = []
    if mode == 'full_warehouse':
        if pd.notna(row.get('visible_queries')) and row['visible_queries'] < test['visible_queries'].median(): reasons.append('narrow_query_coverage')
        if pd.notna(row.get('top_query_share')) and row['top_query_share'] > test['top_query_share'].median(): reasons.append('concentrated_query_mix')
        if row['imp_prev30'] < test['imp_prev30'].median(): reasons.append('lower_recent_visibility')
    else:
        if row['days_since_last_update'] >= 180: reasons.append('stale_visible_page')
        if row['avg_position'] <= 20 and row['ctr_calc'] < 0.005: reasons.append('low_ctr_visible_page')
        if row['avg_position'] <= 10 and row['content_age_days'] >= 180: reasons.append('page_one_decay_risk')
        if row['word_count'] < 1200 and row['impressions_90d'] >= 250: reasons.append('thin_visible_page')
    return '|'.join(reasons) if reasons else 'general_review'

scored['reason_codes'] = scored.apply(make_reason, axis=1)
scored['action'] = np.where(scored['reason_codes'].str.contains('low_ctr'), 'refresh_and_review_ctr',
                      np.where(scored['reason_codes'].str.contains('stale|decay'), 'refresh', 'review'))
scored = scored.sort_values('model_score', ascending=False).reset_index(drop=True)
scored['rank'] = np.arange(1, len(scored)+1)

keep = [id_col, group_col, 'rank', 'model_score', 'baseline_score', 'reason_codes', 'action', 'target']
for c in feature_cols:
    if c not in keep: keep.append(c)
top10 = scored[keep].head(10).copy()
top10['what_could_make_it_wrong'] = np.where(
    top10['reason_codes'].str.contains('ctr'),
    'CTR can reflect query intent or SERP presentation rather than a content problem.',
    'The observed signal may be temporary, so the page should be reviewed before changing it.'
)

display(top10)
top10.to_csv(ROOT / 'work' / 'outputs' / 'capstone_top10_recommendations.csv', index=False)
scored.head(1000).to_csv(ROOT / 'work' / 'outputs' / 'capstone_ranked_recommendations_top1000.csv', index=False)


## 6. Leakage and validation checks

The target is never included in the feature list. The holdout is grouped by client so the model is tested on clients that were not used for training. This is intentionally stricter than a random row split because content rows from the same client can otherwise make the task look easier than it is.

In [ ]:
leakage_names = {'target','trend_direction','trend_pct','is_declining_label','imp_next30'}
leakage_in_features = sorted(leakage_names.intersection(feature_cols))
checks = {
    'leakage_features_present': leakage_in_features,
    'grouped_holdout': True,
    'train_rows': int(len(train)),
    'test_rows': int(len(test)),
    'train_groups': int(train[group_col].nunique()),
    'test_groups': int(test[group_col].nunique()),
    'overlapping_groups': int(len(set(train[group_col]) & set(test[group_col]))),
}
print(json.dumps(checks, indent=2))
assert not leakage_in_features
assert checks['overlapping_groups'] == 0
print('LEAKAGE / GROUP SPLIT CHECK: PASS')

## 7. Create the research-paper data

The notebook writes a compact JSON summary and a public-safe HTML research paper. The HTML is intentionally self-contained so it can be deployed directly with GitHub Pages.

In [ ]:
def pct(x): return f'{100*x:.1f}%'
result_payload = {
    'title': 'Search Decline Signals and a Transparent Content Review Queue',
    'mode': mode,
    'decision_date': decision_date,
    'rows': int(len(data)),
    'test_rows': int(len(test)),
    'base_rate': float(data['target'].mean()),
    'metrics': metrics,
    'top10': top10.to_dict(orient='records'),
    'feature_cols': feature_cols,
    'signal_summary': signal_summary,
    'leakage_check': checks,
}
(ROOT / 'paper' / 'results.json').write_text(json.dumps(result_payload, indent=2, default=str))

def bar(label, value, max_value=1.0):
    width = max(0, min(100, 100*value/max_value))
    return f'<div class="barrow"><span>{label}</span><div class="bar"><i style="width:{width:.1f}%"></i></div><b>{value:.3f}</b></div>'

m = metrics
html = f'''<!doctype html>
<html><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>Search Decline Signals and a Transparent Content Review Queue</title>
<style>
body{{font-family:Inter,Arial,sans-serif;line-height:1.65;color:#102a2e;background:#f6faf9;margin:0}}
.wrap{{max-width:1050px;margin:auto;padding:42px 22px 80px}} .hero{{background:#fff;padding:42px;border-radius:20px;box-shadow:0 8px 30px #0b303014}}
h1{{font-size:42px;line-height:1.12;margin:0 0 14px}} h2{{margin-top:42px}} h3{{margin-top:28px}}
.tag{{display:inline-block;padding:6px 11px;border-radius:999px;background:#dff4ea;margin:3px;font-size:13px}}
.grid{{display:grid;grid-template-columns:repeat(auto-fit,minmax(210px,1fr));gap:14px;margin:20px 0}}
.card{{background:#fff;border:1px solid #dbe8e4;border-radius:14px;padding:18px}} .big{{font-size:28px;font-weight:700}}
table{{width:100%;border-collapse:collapse;background:#fff}} th,td{{padding:10px;border-bottom:1px solid #dbe8e4;text-align:left;font-size:14px}}
th{{background:#edf6f3}} .barrow{{display:grid;grid-template-columns:170px 1fr 70px;gap:10px;align-items:center;margin:10px 0;font-size:13px}}
.bar{{height:12px;background:#e6efed;border-radius:10px;overflow:hidden}} .bar i{{display:block;height:100%;background:#0d7377}}
.note{{padding:16px 18px;background:#eef8f4;border-left:4px solid #0d7377;border-radius:8px}}
.small{{color:#52686b;font-size:13px}} code{{background:#edf1f0;padding:2px 5px;border-radius:4px}}
</style></head><body><main class="wrap"><section class="hero">
<div><span class="tag">FlyRank ML Capstone</span><span class="tag">Search Intelligence</span><span class="tag">Client-grouped validation</span></div>
<h1>Search Decline Signals and a Transparent Content Review Queue</h1>
<p><b>Research question:</b> Which observable search-performance signals are associated with future content decline, and can they support a transparent ranking system for deciding what to review first?</p>
<p class="small">Data mode: <b>{mode}</b> · Analysis rows: <b>{len(data):,}</b> · Holdout rows: <b>{len(test):,}</b></p>
</section>
<section><h2>Abstract</h2><p>This study tests whether simple search-performance signals can identify pages that deserve earlier review when visibility declines. I first examine descriptive relationships between decline, freshness, CTR and search position, then compare a transparent hand-written baseline with a Random Forest model using the same client-grouped holdout. The result is an explainable review queue with reason codes rather than an automatic content-change system. The findings are directional and decision-support oriented: they do not establish causation or reveal Google's ranking algorithm.</p></section>
<section><h2>Research decision</h2><p>The practical decision supported by this work is <b>which pages should a content team inspect first?</b> The system is not designed to decide that a page must be rewritten. Human review remains the final step.</p></section>
<section><h2>Data and validation</h2><div class="grid">
<div class="card"><div class="big">{len(data):,}</div><div>analysis rows</div></div>
<div class="card"><div class="big">{pct(data['target'].mean())}</div><div>decline base rate</div></div>
<div class="card"><div class="big">{checks['train_groups']}</div><div>training clients/groups</div></div>
<div class="card"><div class="big">{checks['test_groups']}</div><div>holdout clients/groups</div></div>
</div><p>The preferred warehouse path uses a pre-decision feature window and a future 30-day outcome. The model and baseline are evaluated on the same client-grouped holdout. No label-derived field is used as a feature.</p></section>
<section><h2>Model versus transparent baseline</h2><div class="grid">
<div class="card"><b>Precision@50</b>{bar('Baseline',m['baseline_precision_at_50'])}{bar('Random Forest',m['model_precision_at_50'])}</div>
<div class="card"><b>Precision@100</b>{bar('Baseline',m['baseline_precision_at_100'])}{bar('Random Forest',m['model_precision_at_100'])}</div>
<div class="card"><b>Average precision</b>{bar('Baseline',m['baseline_average_precision'])}{bar('Random Forest',m['model_average_precision'])}</div>
</div><p>ROC AUC for the model is <b>{m['model_roc_auc']:.3f}</b>. These measurements describe this holdout only; they are not future-performance guarantees.</p></section>
<section><h2>What the signals say</h2><p>{signal_summary['freshness']}. {signal_summary['ctr_position']}.</p><div class="note">A signal is treated as evidence for prioritization, not as proof of causality. Search intent, SERP features, seasonality and other unobserved factors can change CTR or visibility.</div></section>
<section><h2>Top-10 review queue</h2><table><tr><th>Rank</th><th>Action</th><th>Score</th><th>Reason</th><th>Observed label</th></tr>'''
for _, r in top10.iterrows():
    html += f"<tr><td>{int(r['rank'])}</td><td>{r['action']}</td><td>{float(r['model_score']):.4f}</td><td>{r['reason_codes']}</td><td>{int(r['target'])}</td></tr>"
html += '''</table><p class="small">A high rank means “review earlier,” not “change automatically.” Each recommendation should be checked against query intent, current SERP presentation and the actual page.</p></section>
<section><h2>Limitations and honest framing</h2><ul><li>Association is not causation.</li><li>Client-grouped validation reduces client memorization but does not guarantee performance on unseen businesses.</li><li>The public starter mode is a smaller safe slice; the preferred capstone run uses the gated warehouse.</li><li>CTR is affected by query intent and SERP presentation, so low CTR alone is not a content diagnosis.</li><li>The ranking queue is decision support and should not be interpreted as proof of Google's algorithm.</li></ul></section>
<section><h2>Reproducibility</h2><p>The complete analysis is in <code>work/notebooks/capstone_analysis.ipynb</code>. The notebook writes the ranked outputs and this paper from the same run. No private queries, client names, credentials or raw gated exports are included in the repository.</p></section>
<section><h2>Data credit</h2><p>Built on the FlyRank ML Internship search-performance warehouse and its public-safe starter material. The dataset is used for research and internship practice; the analysis does not claim to reveal proprietary ranking factors.</p></section>
</main></body></html>'''
(ROOT / 'paper' / 'index.html').write_text(html, encoding='utf-8')
print('Wrote paper/index.html and paper/results.json')
print('FINAL CAPSTONE SELF-CHECK: PASS')